In [3]:
import pandas as pd

tm = pd.read_csv("../data/scrape/pro/player_stats.csv")
tm_matches = pd.read_csv("../data/scrape/pro/matches.csv")
tm_players = pd.read_csv("../data/scrape/pro/players.csv")

tm_players = tm_players[["player_id", "player_name", "position"]]
tm_matches = tm_matches[["match_id", "date"]]

tm = tm.merge(tm_players, on="player_id", how="left")
tm = tm.merge(tm_matches, on="match_id", how="left")

ss = pd.read_csv("../data/sofascore/ratings.csv")
ss = ss.rename(columns={"datum": "date"})
ss = ss.rename(columns={"name": "player_name"})

In [ ]:
def merge_and_split(tm, ss, df_matched, on_cols):
    """
    Führt Merge durch und:
    - hängt erfolgreiche Matches an df_matched an
    - entfernt gematchte Rows aus tm
    - gibt Anzahl neuer Matches aus
    """

    # 1. Merge mit Indicator
    merged = tm.merge(
        ss,
        on=on_cols,
        how="left",
        indicator=True
    )

    # 2. Erfolgreiche Matches
    new_matches = merged[merged["_merge"] == "both"].drop(columns="_merge")

    # 👉 Anzahl neue Matches
    n_new = len(new_matches)

    # 3. An bestehende Matches anhängen
    df_matched = pd.concat([df_matched, new_matches], ignore_index=True)

    # 4. Matches aus tm entfernen
    tm_new = tm.merge(
        new_matches[on_cols].drop_duplicates(),
        on=on_cols,
        how="left",
        indicator=True
    )

    tm_new = tm_new[tm_new["_merge"] == "left_only"].drop(columns="_merge")

    # 👉 Print für Monitoring
    print(f"{n_new} neue Spieler gematcht")

    return tm_new, df_matched